# Olympus OM-D E-M1x GPS log evaluation

In [ ]:
# imports
from ipyleaflet import Map, Marker

import pynmea2
import pandas
from datetime import datetime as dt
from geodata import GeoDataPoint as gdp

In [ ]:
# basic map initialization

center = (52.283935, 8.022916)
map = Map(center=center, zoom=13)

marker = Marker(location=center)
map.add_control(marker)

display(map)

In [ ]:
# parse Olympus GPS Log data
pathToGpsLog = "./path/to/GPS.LOG"

with open(pathToGpsLog) as gpsFile:
    lines = gpsFile.readlines()

data = []

for line in lines:
    try:
        message = pynmea2.parse(line)

        if message.sentence_type == "RMC":
            dateTime = dt.combine(message.datestamp, message.timestamp)
            
            geodatapoint = gdp(dateTime, message.latitude, message.longitude)
            
            data.append(geodatapoint)

    except pynmea2.ParseError:
        continue

In [ ]:
for datapoint in data:
	marker = Marker(location=datapoint.coordinate, draggable=False, title=f"Lat: {datapoint.latitude}, Lon: {datapoint.longitude}")
	map.add_control(marker)

display(map)

In [ ]:
# parse Olympus sensor data
pathToSnsLog = "./path/to/LOG.SNS"

rows = []
with open(pathToSnsLog) as gpsFile:
    current_time = None
    for line in gpsFile:
        line = line.strip()
        # Time
        if line.startswith("$OLTIM"):
            _, date, time = line.split(",")
            current_time = dateTime.strptime(date + time, "%Y%m%d%H%M%S")
        # Compass
        elif line.startswith("$OLCMP"):
            rows[0]["compass_deg"] = float(line.split(",")[1])
        # Pressure
        elif line.startswith("$OLPRE"):
            p = line.split(",")
            rows[0]["pressure_hpa"] = float(p[1])
        # Temperature
        elif line.startswith("$OLTMP"):
            t = line.split(",")
            rows[0]["temp_c"] = float(t[1])
        # Accelerometer
        elif line.startswith("$OLACC"):
            a = line.split(",")
            rows[0]["acc_x"] = float(a[1])
            rows[0]["acc_y"] = float(a[2])
            rows[0]["acc_z"] = float(a[3])
        # Start a new record
        elif line.startswith("$OLTIM"):
            rows.append({"datetime": current_time})

dataframe = pandas.DataFrame(rows)
dataframe.head()